In [14]:
import json
import os
import sys
import numpy as np
import uuid
import pandas as pd
import textwrap
import re
from IPython import InteractiveShell
InteractiveShell.ast_node_interactivity="all"
import pandas as pd
import plotly.graph_objects as go
import plotly.colors

In [7]:
comms_df=pd.read_csv("data/MC3_data_no_pseudonyms.csv")
comms_df.head()

,content,timestamp,date,timeblock,source_entity_type,target_entity_type,source,target,hour,time,mins_since_start_of_day,packet_id,Message_ID,Processed_Text,Entities
0,"hey the intern, it's the lookout! just spotted...",01/10/40 8:09,01/10/40,8:15,Person,Person,the lookout,the intern,8,8:09:00,0,1,1,just spotted a pod of dolphins near the easter...,[]
1,"hey the lookout, the intern here! i'd absolute...",01/10/40 8:10,01/10/40,8:15,Person,Person,the intern,the lookout,8,8:10:00,1,1,2,i'd absolutely love to join you for birdwatchi...,['mrs. money']
2,"sam, it's kelly! let's meet at sunrise point a...",01/10/40 8:13,01/10/40,8:15,Person,Person,kelly,sam,8,8:13:00,4,1,3,let's meet at sunrise point at 7 am for birdwa...,['sunrise point']
3,"mrs. money, it's the intern. just checking in ...",01/10/40 8:16,01/10/40,8:30,Person,Person,the intern,mrs. money,8,8:16:00,7,1,4,just checking in to see what tasks you need he...,['the lookout']
4,"boss, it's mrs. money. i've reviewed our opera...",01/10/40 8:19,01/10/40,8:30,Person,Person,mrs. money,boss,8,8:19:00,10,1,5,i've reviewed our operational funding for the ...,['the middleman']


In [8]:
def round_up_to_15min(dt):
    # If already on a 15-min mark, keep as is
    if dt.minute % 15 == 0 and dt.second == 0 and dt.microsecond == 0:
        return dt
    # Otherwise, round up to next 15-min mark
    nmins = ((dt.minute // 15) + 1) * 15
    if nmins == 60:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(hours=1)
    else:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(minutes=nmins)
    return dt

def round_to_nearest_15min(dt):
    discard = pd.Timedelta(minutes=dt.minute % 15,
                           seconds=dt.second,
                           microseconds=dt.microsecond)
    dt -= discard
    if discard >= pd.Timedelta(minutes=7.5):
        dt += pd.Timedelta(minutes=15)
    return dt

results = []

for packet_id in comms_df['packet_id'].unique():
    df = comms_df.loc[comms_df['packet_id'] == packet_id].copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by=['timestamp', 'Message_ID']).reset_index(drop=True)
    date = df['date'].iloc[0]  # Get the date for this packet
    pair_dict = {}
    for idx, row in df.iterrows():
        src = row['source']
        tgt = row['target']
        ts = row['timestamp']
        pair = (src, tgt)
        rev_pair = (tgt, src)
        if pair not in pair_dict:
            pair_dict[pair] = {'direction': '->', 'latest_ts': ts}
        else:
            if ts > pair_dict[pair]['latest_ts']:
                pair_dict[pair]['latest_ts'] = ts
        if rev_pair in pair_dict:
            pair_dict[pair]['direction'] = '<->'
            pair_dict[rev_pair]['direction'] = '<->'
    seen_pairs = set()
    for (src, tgt), info in pair_dict.items():
        if info['direction'] == '<->' and (tgt, src) in seen_pairs:
            continue
        seen_pairs.add((src, tgt))
        rounded_time = round_to_nearest_15min(info['latest_ts']).strftime('%H:%M')
        
        if info['direction'] == '<->':
            pair_str = f"{src} <-> {tgt}"
        else:
            pair_str = f"{src} -> {tgt}"
        results.append({
            'packet_id': packet_id,
            'date': date,
            'Pair': pair_str,
            'Time block': f"[{rounded_time}]"
        })

# Convert to DataFrame
sunburst_packet_df = pd.DataFrame(results)
sunburst_packet_df.head()

/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_32716/755768800.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'])
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_32716/755768800.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'])
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_32716/755768800.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['timestamp'] = pd.to_datetime(df['timestamp'])
/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn

,packet_id,date,Pair,Time block
0,1,01/10/40,the lookout <-> the intern,[08:15]
1,1,01/10/40,kelly -> sam,[08:15]
2,1,01/10/40,the intern -> mrs. money,[08:15]
3,1,01/10/40,mrs. money <-> boss,[08:30]
4,1,01/10/40,boss <-> the middleman,[08:30]


In [12]:
sunburst_packet_df.loc[sunburst_packet_df['packet_id']==34]

,packet_id,date,Pair,Time block
151,34,06/10/40,green guardians -> reef guardian,[09:45]
152,34,06/10/40,reef guardian -> paackland harbor,[10:00]
153,34,06/10/40,paackland harbor -> mako,[10:00]
154,34,06/10/40,remora -> sailor shifts team,[10:00]
155,34,06/10/40,glitters team -> boss,[10:00]


In [16]:
sunburst_packet_df.to_csv("data/sunburst_packet_df.csv", index=False)

In [15]:
# Assuming output_df is already available from the notebook
# This code should be run after the output_df is created in the notebook

# Define all 15-min time slots (clockwise order)
all_time_slots = pd.date_range("08:00", "15:00", freq="15min").strftime("%H:%M").tolist()[:-1][::-1]

labels = []
parents = []
ids = []
colors = []
customdata = []
hovertemplate = []

gray_color = "lightgray"

# Get unique dates and assign a unique color to each (clockwise)
dates = sorted(sunburst_packet_df['date'].unique(), reverse=True)
palette = plotly.colors.qualitative.Plotly
while len(palette) < len(dates):
    palette += palette

date_color_map = {date: palette[i] for i, date in enumerate(dates)}

# Root node
labels.append("root")
parents.append("")
ids.append("root")
colors.append("white")
customdata.append("")
hovertemplate.append("<b>All Dates</b>")

for date in dates:
    labels.append(str(date))
    parents.append("root")
    ids.append(str(date))
    colors.append(date_color_map[date])
    customdata.append("")
    hovertemplate.append(f"<b>Date:</b> {date}")

    df_date = sunburst_packet_df[sunburst_packet_df['date'] == date]
    
    # Collect all pairs for this date with their time slots
    pair_time_map = {}
    for _, row in df_date.iterrows():
        slots = [s.strip() for s in row['Time block'].strip('[]').split(',')]
        pair = row['Pair']
        if pair not in pair_time_map:
            pair_time_map[pair] = []
        pair_time_map[pair].extend(slots)
    
    # Create a map of time slots to pairs for this date
    slot_to_pairs = {slot: [] for slot in all_time_slots}
    for pair, slots in pair_time_map.items():
        for slot in slots:
            if slot in slot_to_pairs:
                slot_to_pairs[slot].append(pair)
    
    # Add children in clockwise order (all_time_slots is already in clockwise order)
    for slot in all_time_slots:
        pairs_in_slot = slot_to_pairs[slot]
        
        if pairs_in_slot:
            # Add pairs for this time slot
            for pair in sorted(set(pairs_in_slot), reverse=True):
                labels.append(pair)
                parents.append(str(date))
                ids.append(f"{date}-{slot}-{pair}")
                colors.append(date_color_map[date])
                customdata.append(f"{pair}|{slot}")
                hovertemplate.append(f"<b>Pair:</b> {pair}<br><b>Time Slot:</b> {slot}")
        else:
            # No communications in this time slot
            labels.append(f"no comms")
            parents.append(str(date))
            ids.append(f"{date}-{slot}-no-comms")
            colors.append(gray_color)
            customdata.append(slot)
            hovertemplate.append(f"<b>No comms</b><br><b>Time Slot:</b> {slot}")

fig = go.Figure(go.Sunburst(
    labels=labels,
    parents=parents,
    ids=ids,
    branchvalues="total",
    insidetextorientation='radial',
    marker=dict(colors=colors),
    customdata=customdata,
    hovertemplate=hovertemplate,
    sort=False
))

fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0),
    title="Entities (Pairs) by Date with Clockwise Time Slots (No Comms in Gray)",
    width=900,
    height=900
)

# fig.show()

In [2]:
# import plotly.io as pio
# pio.renderers.default = "notebook"

In [3]:

df=pd.read_csv("data/clustered_messages.csv")

df.head()
df.shape

,content,timestamp,date,timeblock,source_entity_type,target_entity_type,source,target,hour,time,mins_since_start_of_day,packet_id,Message_ID,Processed_Text,Entities,source_entity,target_entity,cluster
0,"hey the intern, it's the lookout! just spotted...",01/10/40 8:09,01/10/40,8:15,Person,Person,the lookout,the intern,8,8:09:00,0,1,1,just spotted a pod of dolphins near the easter...,[],the lookout,the intern,6
1,"hey the lookout, the intern here! i'd absolute...",01/10/40 8:10,01/10/40,8:15,Person,Person,the intern,the lookout,8,8:10:00,1,1,2,i'd absolutely love to join you for birdwatchi...,['mrs. money'],the intern,the lookout,6
2,"sam, it's kelly! let's meet at sunrise point a...",01/10/40 8:13,01/10/40,8:15,Person,Person,kelly,sam,8,8:13:00,4,1,3,let's meet at sunrise point at 7 am for birdwa...,['sunrise point'],kelly,sam,6
3,"mrs. money, it's the intern. just checking in ...",01/10/40 8:16,01/10/40,8:30,Person,Person,the intern,mrs. money,8,8:16:00,7,1,4,just checking in to see what tasks you need he...,['the lookout'],the intern,mrs. money,6
4,"boss, it's mrs. money. i've reviewed our opera...",01/10/40 8:19,01/10/40,8:30,Person,Person,mrs. money,boss,8,8:19:00,10,1,5,i've reviewed our operational funding for the ...,['the middleman'],mrs. money,boss,8


(580, 18)

In [4]:
# sunburst_df = df.loc[df['packet_id'] <= 20].copy()
sunburst_df=df.copy()
sunburst_df['source'] = sunburst_df['source'].str.lower()
sunburst_df['target'] = sunburst_df['target'].str.lower()
sunburst_df.head()

,content,timestamp,date,timeblock,source_entity_type,target_entity_type,source,target,hour,time,mins_since_start_of_day,packet_id,Message_ID,Processed_Text,Entities,source_entity,target_entity,cluster
0,"hey the intern, it's the lookout! just spotted...",01/10/40 8:09,01/10/40,8:15,Person,Person,the lookout,the intern,8,8:09:00,0,1,1,just spotted a pod of dolphins near the easter...,[],the lookout,the intern,6
1,"hey the lookout, the intern here! i'd absolute...",01/10/40 8:10,01/10/40,8:15,Person,Person,the intern,the lookout,8,8:10:00,1,1,2,i'd absolutely love to join you for birdwatchi...,['mrs. money'],the intern,the lookout,6
2,"sam, it's kelly! let's meet at sunrise point a...",01/10/40 8:13,01/10/40,8:15,Person,Person,kelly,sam,8,8:13:00,4,1,3,let's meet at sunrise point at 7 am for birdwa...,['sunrise point'],kelly,sam,6
3,"mrs. money, it's the intern. just checking in ...",01/10/40 8:16,01/10/40,8:30,Person,Person,the intern,mrs. money,8,8:16:00,7,1,4,just checking in to see what tasks you need he...,['the lookout'],the intern,mrs. money,6
4,"boss, it's mrs. money. i've reviewed our opera...",01/10/40 8:19,01/10/40,8:30,Person,Person,mrs. money,boss,8,8:19:00,10,1,5,i've reviewed our operational funding for the ...,['the middleman'],mrs. money,boss,8


In [5]:
sunburst_df

,content,timestamp,date,timeblock,source_entity_type,target_entity_type,source,target,packet_id,Message_ID,Processed_Text,Entities,source_entity,target_entity,cluster
0,"hey the intern, it's the lookout! just spotted...",2040-01-10 08:09:00,2040-01-10,8:15,Person,Person,the lookout,the intern,1,1,just spotted a pod of dolphins near the easter...,[],the lookout,the intern,6
1,"hey the lookout, the intern here! i'd absolute...",2040-01-10 08:10:00,2040-01-10,8:15,Person,Person,the intern,the lookout,1,2,i'd absolutely love to join you for birdwatchi...,['mrs. money'],the intern,the lookout,6
2,"sam, it's kelly! let's meet at sunrise point a...",2040-01-10 08:13:00,2040-01-10,8:15,Person,Person,kelly,sam,1,3,let's meet at sunrise point at 7 am for birdwa...,['sunrise point'],kelly,sam,6
3,"mrs. money, it's the intern. just checking in ...",2040-01-10 08:16:00,2040-01-10,8:30,Person,Person,the intern,mrs. money,1,4,just checking in to see what tasks you need he...,['the lookout'],the intern,mrs. money,6
4,"boss, it's mrs. money. i've reviewed our opera...",2040-01-10 08:19:00,2040-01-10,8:30,Person,Person,mrs. money,boss,1,5,i've reviewed our operational funding for the ...,['the middleman'],mrs. money,boss,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,v. miesel hq to neptune. harbor closure timeli...,2040-12-10 12:52:00,2040-12-10,13:00,Organization,Vessel,v. miesel shipping,neptune,86,576,your emergency session request is denied. our ...,['oceanus environmental department'],v. miesel shipping,neptune,4
576,neptune to remora. final confirmation: operati...,2040-12-10 12:55:00,2040-12-10,13:00,Vessel,Vessel,neptune,remora,86,577,your video evidence has been invaluable. pleas...,"['nemo reef', 'oceanus city council']",neptune,remora,2
577,"davis, remora here. what's your eta with the a...",2040-12-10 12:56:00,2040-12-10,13:00,Vessel,Person,remora,davis,86,578,be advised that conservation vessel reef guard...,"['reef guardian', 'nemo reef']",remora,davis,5
578,defender to reef guardian. your access denial ...,2040-12-10 12:57:00,2040-12-10,13:00,Vessel,Vessel,defender,reef guardian,86,579,proceed to southern dock at 2200 for equipment...,"[""himark's south dock"", 'boss', 'davis']",defender,reef guardian,9


In [6]:
# sunburst_packet_df=sunburst_df.loc[sunburst_df['packet_id']==1].loc[:,['source','target','Message_ID']]

sunburst_packet_df=sunburst_df.loc[:,['source','target','packet_id','Message_ID','timestamp','date']]

In [7]:
sunburst_packet_df['date']=pd.to_datetime(sunburst_packet_df['date'])
sunburst_packet_df['date']=sunburst_packet_df['date'].dt.strftime('%Y-%m-%d')

sunburst_packet_df.head()

,source,target,packet_id,Message_ID,timestamp,date
0,the lookout,the intern,1,1,2040-01-10 08:09:00,2040-01-10
1,the intern,the lookout,1,2,2040-01-10 08:10:00,2040-01-10
2,kelly,sam,1,3,2040-01-10 08:13:00,2040-01-10
3,the intern,mrs. money,1,4,2040-01-10 08:16:00,2040-01-10
4,mrs. money,boss,1,5,2040-01-10 08:19:00,2040-01-10


In [8]:
# sunburst_packet_df=sunburst_packet_df.loc[sunburst_packet_df['date']<='2040-10-03']

In [9]:
sunburst_packet_df

,source,target,packet_id,Message_ID,timestamp,date
0,the lookout,the intern,1,1,2040-01-10 08:09:00,2040-01-10
1,the intern,the lookout,1,2,2040-01-10 08:10:00,2040-01-10
2,kelly,sam,1,3,2040-01-10 08:13:00,2040-01-10
3,the intern,mrs. money,1,4,2040-01-10 08:16:00,2040-01-10
4,mrs. money,boss,1,5,2040-01-10 08:19:00,2040-01-10
...,...,...,...,...,...,...
575,v. miesel shipping,neptune,86,576,2040-12-10 12:52:00,2040-12-10
576,neptune,remora,86,577,2040-12-10 12:55:00,2040-12-10
577,remora,davis,86,578,2040-12-10 12:56:00,2040-12-10
578,defender,reef guardian,86,579,2040-12-10 12:57:00,2040-12-10


In [10]:
##SNIPPET: TO APPLY ENTITY RECONCILIATION
entity_mapper=pd.read_excel('data/unique_entities.xlsx')
entity_mapper['entity']=entity_mapper['entity'].str.lower()
entity_mapper['recoinciled_entity']=entity_mapper['recoinciled_entity'].str.lower() 
##If toggle is selected, then apply entity reconciliation
sunburst_packet_df['source']=sunburst_packet_df['source'].map(entity_mapper.set_index('entity')['recoinciled_entity'])
sunburst_packet_df['target']=sunburst_packet_df['target'].map(entity_mapper.set_index('entity')['recoinciled_entity'])

In [11]:
import numpy as np

# Ensure timestamp is datetime
sunburst_packet_df['timestamp'] = pd.to_datetime(sunburst_packet_df['timestamp'])

def round_up_to_15min(dt):
    # If already on a 15-min mark, keep as is
    if dt.minute % 15 == 0 and dt.second == 0 and dt.microsecond == 0:
        return dt
    # Otherwise, round up to next 15-min mark
    nmins = ((dt.minute // 15) + 1) * 15
    if nmins == 60:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(hours=1)
    else:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(minutes=nmins)
    return dt

sunburst_packet_df['timeblock'] = sunburst_packet_df['timestamp'].apply(round_up_to_15min)
sunburst_packet_df['timeblock'] = sunburst_packet_df['timeblock'].dt.strftime('%H:%M')

sunburst_packet_df

,source,target,packet_id,Message_ID,timestamp,date,timeblock
0,kelly,sam,1,1,2040-01-10 08:09:00,2040-01-10,08:15
1,sam,kelly,1,2,2040-01-10 08:10:00,2040-01-10,08:15
2,kelly,sam,1,3,2040-01-10 08:13:00,2040-01-10,08:15
3,sam,elise,1,4,2040-01-10 08:16:00,2040-01-10,08:30
4,elise,nadia conti,1,5,2040-01-10 08:19:00,2040-01-10,08:30
...,...,...,...,...,...,...,...
575,v. miesel shipping,neptune,86,576,2040-12-10 12:52:00,2040-12-10,13:00
576,neptune,remora,86,577,2040-12-10 12:55:00,2040-12-10,13:00
577,remora,davis,86,578,2040-12-10 12:56:00,2040-12-10,13:00
578,defender,reef guardian,86,579,2040-12-10 12:57:00,2040-12-10,13:00


In [12]:
sunburst_packet_df.to_csv("data/sunburst_packet_df.csv",index=False)

In [13]:
# import pandas as pd
# import plotly.graph_objects as go

# # Define 30-min intervals from 08:00 to 14:30
# intervals = pd.date_range("08:00", "15:00", freq="30min").strftime("%H:%M").tolist()
# intervals = intervals[:-1]  # Exclude 15:00, so last is 14:30

# # Explicitly reorder so "08:00" is first
# intervals = [i for i in intervals if i != "08:00"]
# intervals = ["08:00"] + intervals
# # Ensure intervals are in correct order (just in case)
# intervals = sorted(intervals, key=lambda x: int(x[:2])*60 + int(x[3:]))


# labels = ["start"]
# parents = [""]
# ids = ["start"]

# num_placeholders = 1  # or any number you want

# for interval in intervals:
#     labels.append(interval)
#     parents.append("start")
#     ids.append(interval)
#     for i in range(1, num_placeholders + 1):
#         labels.append(f"{interval}-placeholder-{i}")
#         parents.append(interval)
#         ids.append(f"{interval}-placeholder-{i}")

# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     ids=ids,
#     branchvalues="total",
#     insidetextorientation='radial',
#     sort=False  # This is critical!
# ))

# fig.update_layout(
#     margin=dict(t=0, l=0, r=0, b=0),
#     title="Sunburst: Time Intervals as Rings with Placeholders (08:00 at Top)",
#     width=900,
#     height=900
# )
# # fig.show()

In [29]:
sunburst_packet_df_filtered=sunburst_packet_df.loc[sunburst_packet_df['packet_id']==1]

In [30]:
sunburst_packet_df_filtered

,source,target,packet_id,Message_ID,timestamp,date,timeblock
0,kelly,sam,1,1,2040-01-10 08:09:00,2040-01-10,08:15
1,sam,kelly,1,2,2040-01-10 08:10:00,2040-01-10,08:15
2,kelly,sam,1,3,2040-01-10 08:13:00,2040-01-10,08:15
3,sam,elise,1,4,2040-01-10 08:16:00,2040-01-10,08:30
4,elise,nadia conti,1,5,2040-01-10 08:19:00,2040-01-10,08:30
5,nadia conti,elise,1,6,2040-01-10 08:21:00,2040-01-10,08:30
6,elise,nadia conti,1,7,2040-01-10 08:24:00,2040-01-10,08:30
7,nadia conti,liam thorne,1,8,2040-01-10 08:26:00,2040-01-10,08:30
8,liam thorne,nadia conti,1,9,2040-01-10 08:29:00,2040-01-10,08:30
9,nadia conti,liam thorne,1,10,2040-01-10 08:32:00,2040-01-10,08:45


In [17]:
sunburst_packet_df_filtered=sunburst_packet_df.loc[sunburst_packet_df['packet_id']==1]

In [18]:
sunburst_packet_df_filtered

##Expected output:
## Pair: the lookout <-> the intern  Time block: [08:15]
## Pair:kelly <-> sam, Time block: [08:15]
## Pair: the intern -> mrs. money, Time block: [08:30]
## Pair: mrs. money <-> boss <-> the middleman , Time block: [08:30,08:45]

,packet_id,date,Pair,Time block
0,1,01/10/40,the lookout <-> the intern,[08:15]
1,1,01/10/40,kelly -> sam,[08:15]
2,1,01/10/40,the intern -> mrs. money,[08:15]
3,1,01/10/40,mrs. money <-> boss,[08:30]
4,1,01/10/40,boss <-> the middleman,[08:30]


In [17]:
# import pandas as pd
# import numpy as np

# def round_to_nearest_15min(dt):
#     # Round to nearest 15 minutes
#     discard = pd.Timedelta(minutes=dt.minute % 15,
#                            seconds=dt.second,
#                            microseconds=dt.microsecond)
#     dt -= discard
#     if discard >= pd.Timedelta(minutes=7.5):
#         dt += pd.Timedelta(minutes=15)
#     return dt

# # Ensure timestamp is datetime
# df = sunburst_packet_df_filtered.copy()
# df['timestamp'] = pd.to_datetime(df['timestamp'])

# # Sort by timestamp and Message_ID if needed
# df = df.sort_values(by=['timestamp', 'Message_ID']).reset_index(drop=True)

# # Track pairs and their directions
# pair_dict = {}

# for idx, row in df.iterrows():
#     src = row['source']
#     tgt = row['target']
#     ts = row['timestamp']
#     pair = (src, tgt)
#     rev_pair = (tgt, src)
#     # If this pair is new, add it
#     if pair not in pair_dict:
#         pair_dict[pair] = {'direction': '->', 'latest_ts': ts}
#     else:
#         # Update latest timestamp if newer
#         if ts > pair_dict[pair]['latest_ts']:
#             pair_dict[pair]['latest_ts'] = ts
#     # If reverse pair exists, mark both as two-way
#     if rev_pair in pair_dict:
#         pair_dict[pair]['direction'] = '<->'
#         pair_dict[rev_pair]['direction'] = '<->'

# # Prepare output
# output = []
# seen_pairs = set()
# for (src, tgt), info in pair_dict.items():
#     # Only print each two-way pair once
#     if info['direction'] == '<->' and (tgt, src) in seen_pairs:
#         continue
#     seen_pairs.add((src, tgt))
#     rounded_time = round_to_nearest_15min(info['latest_ts']).strftime('%H:%M')
#     if info['direction'] == '<->':
#         output.append(f"Pair: {src} <-> {tgt}, Time block: [{rounded_time}]")
#     else:
#         output.append(f"Pair: {src} -> {tgt}, Time block: [{rounded_time}]")

# # Print output
# for line in output:
#     print(line)

In [18]:
# sunburst_packet_df_filtered=sunburst_packet_df.loc[sunburst_packet_df['packet_id']==7]

# import pandas as pd
# import numpy as np

# def round_to_nearest_15min(dt):
#     # Round to nearest 15 minutes
#     discard = pd.Timedelta(minutes=dt.minute % 15,
#                            seconds=dt.second,
#                            microseconds=dt.microsecond)
#     dt -= discard
#     if discard >= pd.Timedelta(minutes=7.5):
#         dt += pd.Timedelta(minutes=15)
#     return dt

# def round_up_to_15min(dt):
#     # If already on a 15-min mark, keep as is
#     if dt.minute % 15 == 0 and dt.second == 0 and dt.microsecond == 0:
#         return dt
#     # Otherwise, round up to next 15-min mark
#     nmins = ((dt.minute // 15) + 1) * 15
#     if nmins == 60:
#         dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(hours=1)
#     else:
#         dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(minutes=nmins)
#     return dt

# # Ensure timestamp is datetime
# df = sunburst_packet_df_filtered.copy()
# df['timestamp'] = pd.to_datetime(df['timestamp'])

# # Sort by timestamp and Message_ID if needed
# df = df.sort_values(by=['timestamp', 'Message_ID']).reset_index(drop=True)

# # Track pairs and their directions
# pair_dict = {}

# for idx, row in df.iterrows():
#     src = row['source']
#     tgt = row['target']
#     ts = row['timestamp']
#     pair = (src, tgt)
#     rev_pair = (tgt, src)
#     # If this pair is new, add it
#     if pair not in pair_dict:
#         pair_dict[pair] = {'direction': '->', 'latest_ts': ts}
#     else:
#         # Update latest timestamp if newer
#         if ts > pair_dict[pair]['latest_ts']:
#             pair_dict[pair]['latest_ts'] = ts
#     # If reverse pair exists, mark both as two-way
#     if rev_pair in pair_dict:
#         pair_dict[pair]['direction'] = '<->'
#         pair_dict[rev_pair]['direction'] = '<->'

# # Prepare output
# output = []
# seen_pairs = set()
# for (src, tgt), info in pair_dict.items():
#     # Only print each two-way pair once
#     if info['direction'] == '<->' and (tgt, src) in seen_pairs:
#         continue
#     seen_pairs.add((src, tgt))
#     rounded_time = round_to_nearest_15min(info['latest_ts']).strftime('%H:%M')
#     if info['direction'] == '<->':
#         output.append(f"Pair: {src} <-> {tgt}, Time block: [{rounded_time}]")
#     else:
#         output.append(f"Pair: {src} -> {tgt}, Time block: [{rounded_time}]")

# # Print output
# for line in output:
#     print(line)

In [19]:
# import pandas as pd
# import numpy as np

# def round_to_nearest_15min(dt):
#     discard = pd.Timedelta(minutes=dt.minute % 15,
#                            seconds=dt.second,
#                            microseconds=dt.microsecond)
#     dt -= discard
#     if discard >= pd.Timedelta(minutes=7.5):
#         dt += pd.Timedelta(minutes=15)
#     return dt

# results = []

# for packet_id in sunburst_packet_df['packet_id'].unique():
#     df = sunburst_packet_df.loc[sunburst_packet_df['packet_id'] == packet_id].copy()
#     df['timestamp'] = pd.to_datetime(df['timestamp'])
#     df = df.sort_values(by=['timestamp', 'Message_ID']).reset_index(drop=True)
#     pair_dict = {}
#     for idx, row in df.iterrows():
#         src = row['source']
#         tgt = row['target']
#         ts = row['timestamp']
#         pair = (src, tgt)
#         rev_pair = (tgt, src)
#         if pair not in pair_dict:
#             pair_dict[pair] = {'direction': '->', 'latest_ts': ts}
#         else:
#             if ts > pair_dict[pair]['latest_ts']:
#                 pair_dict[pair]['latest_ts'] = ts
#         if rev_pair in pair_dict:
#             pair_dict[pair]['direction'] = '<->'
#             pair_dict[rev_pair]['direction'] = '<->'
#     seen_pairs = set()
#     for (src, tgt), info in pair_dict.items():
#         if info['direction'] == '<->' and (tgt, src) in seen_pairs:
#             continue
#         seen_pairs.add((src, tgt))
#         rounded_time = round_to_nearest_15min(info['latest_ts']).strftime('%H:%M')
#         if info['direction'] == '<->':
#             pair_str = f"{src} <-> {tgt}"
#         else:
#             pair_str = f"{src} -> {tgt}"
#         results.append({
#             'packet_id': packet_id,
#             'Pair': pair_str,
#             'Time block': f"[{rounded_time}]"
#         })

# # Convert to DataFrame
# output_df = pd.DataFrame(results)

In [20]:
import pandas as pd
import numpy as np

def round_up_to_15min(dt):
    # If already on a 15-min mark, keep as is
    if dt.minute % 15 == 0 and dt.second == 0 and dt.microsecond == 0:
        return dt
    # Otherwise, round up to next 15-min mark
    nmins = ((dt.minute // 15) + 1) * 15
    if nmins == 60:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(hours=1)
    else:
        dt = dt.replace(minute=0, second=0, microsecond=0) + pd.Timedelta(minutes=nmins)
    return dt

def round_to_nearest_15min(dt):
    discard = pd.Timedelta(minutes=dt.minute % 15,
                           seconds=dt.second,
                           microseconds=dt.microsecond)
    dt -= discard
    if discard >= pd.Timedelta(minutes=7.5):
        dt += pd.Timedelta(minutes=15)
    return dt

results = []

for packet_id in sunburst_packet_df['packet_id'].unique():
    df = sunburst_packet_df.loc[sunburst_packet_df['packet_id'] == packet_id].copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by=['timestamp', 'Message_ID']).reset_index(drop=True)
    date = df['date'].iloc[0]  # Get the date for this packet
    pair_dict = {}
    for idx, row in df.iterrows():
        src = row['source']
        tgt = row['target']
        ts = row['timestamp']
        pair = (src, tgt)
        rev_pair = (tgt, src)
        if pair not in pair_dict:
            pair_dict[pair] = {'direction': '->', 'latest_ts': ts}
        else:
            if ts > pair_dict[pair]['latest_ts']:
                pair_dict[pair]['latest_ts'] = ts
        if rev_pair in pair_dict:
            pair_dict[pair]['direction'] = '<->'
            pair_dict[rev_pair]['direction'] = '<->'
    seen_pairs = set()
    for (src, tgt), info in pair_dict.items():
        if info['direction'] == '<->' and (tgt, src) in seen_pairs:
            continue
        seen_pairs.add((src, tgt))
        rounded_time = round_to_nearest_15min(info['latest_ts']).strftime('%H:%M')
        
        if info['direction'] == '<->':
            pair_str = f"{src} <-> {tgt}"
        else:
            pair_str = f"{src} -> {tgt}"
        results.append({
            'packet_id': packet_id,
            'date': date,
            'Pair': pair_str,
            'Time block': f"[{rounded_time}]"
        })

# Convert to DataFrame
output_df = pd.DataFrame(results)

In [21]:
output_df

,packet_id,date,Pair,Time block
0,1,2040-01-10,kelly <-> sam,[08:15]
1,1,2040-01-10,sam -> elise,[08:15]
2,1,2040-01-10,elise <-> nadia conti,[08:30]
3,1,2040-01-10,nadia conti <-> liam thorne,[08:30]
4,2,2040-01-10,serenity <-> mako,[08:45]
...,...,...,...,...
385,86,2040-12-10,v. miesel shipping -> neptune,[12:45]
386,86,2040-12-10,neptune -> remora,[13:00]
387,86,2040-12-10,remora -> davis,[13:00]
388,86,2040-12-10,defender -> reef guardian,[13:00]


In [22]:
output_df.loc[output_df['packet_id']==2]

,packet_id,date,Pair,Time block
4,2,2040-01-10,serenity <-> mako,[08:45]
5,2,2040-01-10,serenity -> reef guardian,[09:00]
6,2,2040-01-10,himark harbor -> mako,[09:00]
7,2,2040-01-10,mako -> davis,[09:00]


In [23]:
# import pandas as pd
# import plotly.graph_objects as go

# # Assume output_df is already created as per your previous steps

# # Prepare sunburst data
# labels = []
# parents = []
# ids = []

# # Level 0: Root
# labels.append("root")
# parents.append("")
# ids.append("root")

# # Level 1: Dates
# for date in sorted(output_df['date'].unique()):
#     labels.append(str(date))
#     parents.append("root")
#     ids.append(str(date))

#     # Level 2: Time blocks for this date
#     df_date = output_df[output_df['date'] == date]
#     for time_block in sorted(df_date['Time block'].unique()):
#         time_block_clean = time_block.strip("[]")  # Remove brackets
#         time_id = f"{date}-{time_block_clean}"
#         labels.append(time_block_clean)
#         parents.append(str(date))
#         ids.append(time_id)

#         # Level 3: Pairs for this date and time block
#         df_time = df_date[df_date['Time block'] == time_block]
#         for pair in df_time['Pair']:
#             pair_id = f"{date}-{time_block_clean}-{pair}"
#             labels.append(pair)
#             parents.append(time_id)
#             ids.append(pair_id)

# # Create sunburst
# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     ids=ids,
#     branchvalues="total",
#     insidetextorientation='radial',
#     sort=False
# ))

# fig.update_layout(
#     margin=dict(t=0, l=0, r=0, b=0),
#     title="Entities (Pairs) by Date and Time Block",
#     width=900,
#     height=900
# )

# # fig.show()

In [24]:
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.colors

# # Define all 15-min time slots (clockwise order)
# all_time_slots = pd.date_range("08:00", "15:00", freq="15min").strftime("%H:%M").tolist()[:-1][::-1]

# labels = []
# parents = []
# ids = []
# colors = []
# customdata = []
# hovertemplate = []

# gray_color = "lightgray"

# # Get unique dates and assign a unique color to each (clockwise)
# dates = sorted(output_df['date'].unique(), reverse=True)
# palette = plotly.colors.qualitative.Plotly
# while len(palette) < len(dates):
#     palette += palette

# date_color_map = {date: palette[i] for i, date in enumerate(dates)}

# # Root node
# labels.append("root")
# parents.append("")
# ids.append("root")
# colors.append("white")
# customdata.append("")
# hovertemplate.append("<b>All Dates</b>")

# for date in dates:
#     labels.append(str(date))
#     parents.append("root")
#     ids.append(str(date))
#     colors.append(date_color_map[date])
#     customdata.append("")
#     hovertemplate.append(f"<b>Date:</b> {date}")

#     df_date = output_df[output_df['date'] == date]
#     for slot in all_time_slots:
#         time_id = f"{date}-{slot}"
#         labels.append(slot)
#         parents.append(str(date))
#         ids.append(time_id)
#         colors.append("white")  # or use a light color for time slots
#         customdata.append(slot)
#         hovertemplate.append(f"<b>Time:</b> {slot}")

#         # Find pairs for this date and time slot
#         pairs = []
#         for _, row in df_date.iterrows():
#             slots = [s.strip() for s in row['Time block'].strip('[]').split(',')]
#             if slot in slots:
#                 pairs.append(row['Pair'])
#         if pairs:
#             for pair in sorted(set(pairs), reverse=True):
#                 labels.append(pair)
#                 parents.append(time_id)
#                 ids.append(f"{date}-{slot}-{pair}")
#                 colors.append(date_color_map[date])
#                 customdata.append(pair)
#                 hovertemplate.append(f"<b>Pair:</b> {pair}<br><b>Time:</b> {slot}")
#         else:
#             # No comms for this slot
#             labels.append(f"no comms")
#             parents.append(time_id)
#             ids.append(f"{date}-{slot}-no-comms")
#             colors.append(gray_color)
#             customdata.append(slot)
#             hovertemplate.append(f"<b>No comms</b><br><b>Time:</b> {slot}")

# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     ids=ids,
#     branchvalues="total",
#     insidetextorientation='radial',
#     marker=dict(colors=colors),
#     customdata=customdata,
#     hovertemplate=hovertemplate,
#     sort=False
# ))

# fig.update_layout(
#     margin=dict(t=0, l=0, r=0, b=0),
#     title="Entities (Pairs) by Date and 15-Minute Time Block (No Comms in Gray, Clockwise)",
#     width=900,
#     height=900
# )

# # fig.show()

In [25]:
import pandas as pd
import plotly.graph_objects as go
import plotly.colors

# Assuming output_df is already available from the notebook
# This code should be run after the output_df is created in the notebook

# Define all 15-min time slots (clockwise order)
all_time_slots = pd.date_range("08:00", "15:00", freq="15min").strftime("%H:%M").tolist()[:-1][::-1]

labels = []
parents = []
ids = []
colors = []
customdata = []
hovertemplate = []

gray_color = "lightgray"

# Get unique dates and assign a unique color to each (clockwise)
dates = sorted(output_df['date'].unique(), reverse=True)
palette = plotly.colors.qualitative.Plotly
while len(palette) < len(dates):
    palette += palette

date_color_map = {date: palette[i] for i, date in enumerate(dates)}

# Root node
labels.append("root")
parents.append("")
ids.append("root")
colors.append("white")
customdata.append("")
hovertemplate.append("<b>All Dates</b>")

for date in dates:
    labels.append(str(date))
    parents.append("root")
    ids.append(str(date))
    colors.append(date_color_map[date])
    customdata.append("")
    hovertemplate.append(f"<b>Date:</b> {date}")

    df_date = output_df[output_df['date'] == date]
    
    # Collect all pairs for this date with their time slots
    pair_time_map = {}
    for _, row in df_date.iterrows():
        slots = [s.strip() for s in row['Time block'].strip('[]').split(',')]
        pair = row['Pair']
        if pair not in pair_time_map:
            pair_time_map[pair] = []
        pair_time_map[pair].extend(slots)
    
    # Create a map of time slots to pairs for this date
    slot_to_pairs = {slot: [] for slot in all_time_slots}
    for pair, slots in pair_time_map.items():
        for slot in slots:
            if slot in slot_to_pairs:
                slot_to_pairs[slot].append(pair)
    
    # Add children in clockwise order (all_time_slots is already in clockwise order)
    for slot in all_time_slots:
        pairs_in_slot = slot_to_pairs[slot]
        
        if pairs_in_slot:
            # Add pairs for this time slot
            for pair in sorted(set(pairs_in_slot), reverse=True):
                labels.append(pair)
                parents.append(str(date))
                ids.append(f"{date}-{slot}-{pair}")
                colors.append(date_color_map[date])
                customdata.append(f"{pair}|{slot}")
                hovertemplate.append(f"<b>Pair:</b> {pair}<br><b>Time Slot:</b> {slot}")
        else:
            # No communications in this time slot
            labels.append(f"no comms")
            parents.append(str(date))
            ids.append(f"{date}-{slot}-no-comms")
            colors.append(gray_color)
            customdata.append(slot)
            hovertemplate.append(f"<b>No comms</b><br><b>Time Slot:</b> {slot}")

fig = go.Figure(go.Sunburst(
    labels=labels,
    parents=parents,
    ids=ids,
    branchvalues="total",
    insidetextorientation='radial',
    marker=dict(colors=colors),
    customdata=customdata,
    hovertemplate=hovertemplate,
    sort=False
))

fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0),
    title="Entities (Pairs) by Date with Clockwise Time Slots (No Comms in Gray)",
    width=900,
    height=900
)

# fig.show()

In [26]:
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.colors

# # List of entities to highlight
# entities_of_interest = ['kelly', 'sam']  # <-- Edit this list as needed

# # Define all 15-min time slots (clockwise order)
# all_time_slots = pd.date_range("08:00", "15:00", freq="15min").strftime("%H:%M").tolist()[:-1][::-1]

# labels = []
# parents = []
# ids = []
# colors = []
# customdata = []
# hovertemplate = []
# line_colors = []
# line_widths = []

# gray_color = "lightgray"

# # Get unique dates and assign a unique color to each (clockwise)
# dates = sorted(output_df['date'].unique(), reverse=True)
# palette = plotly.colors.qualitative.Plotly
# while len(palette) < len(dates):
#     palette += palette

# date_color_map = {date: palette[i] for i, date in enumerate(dates)}

# # Root node
# labels.append("root")
# parents.append("")
# ids.append("root")
# colors.append("white")
# customdata.append("")
# hovertemplate.append("<b>All Dates</b>")
# line_colors.append("black")
# line_widths.append(1)

# for date in dates:
#     labels.append(str(date))
#     parents.append("root")
#     ids.append(str(date))
#     colors.append(date_color_map[date])
#     customdata.append("")
#     hovertemplate.append(f"<b>Date:</b> {date}")
#     line_colors.append("black")
#     line_widths.append(1)

#     df_date = output_df[output_df['date'] == date]
    
#     # Collect all pairs for this date with their time slots
#     pair_time_map = {}
#     for _, row in df_date.iterrows():
#         slots = [s.strip() for s in row['Time block'].strip('[]').split(',')]
#         pair = row['Pair']
#         if pair not in pair_time_map:
#             pair_time_map[pair] = []
#         pair_time_map[pair].extend(slots)
    
#     # Create a map of time slots to pairs for this date
#     slot_to_pairs = {slot: [] for slot in all_time_slots}
#     for pair, slots in pair_time_map.items():
#         for slot in slots:
#             if slot in slot_to_pairs:
#                 slot_to_pairs[slot].append(pair)
    
#     # Add children in clockwise order (all_time_slots is already in clockwise order)
#     for slot in all_time_slots:
#         pairs_in_slot = slot_to_pairs[slot]
        
#         if pairs_in_slot:
#             # Add pairs for this time slot
#             for pair in sorted(set(pairs_in_slot), reverse=True):
#                 labels.append(pair)
#                 parents.append(str(date))
#                 ids.append(f"{date}-{slot}-{pair}")
#                 colors.append(date_color_map[date])
#                 customdata.append(f"{pair}|{slot}")
#                 hovertemplate.append(f"<b>Pair:</b> {pair}<br><b>Time Slot:</b> {slot}")
#                 # Highlight if any entity of interest is in the pair string
#                 if any(e.lower() in pair.lower() for e in entities_of_interest):
#                     line_colors.append("red")
#                     line_widths.append(4)
#                 else:
#                     line_colors.append("black")
#                     line_widths.append(1)
#         else:
#             # No communications in this time slot
#             labels.append(f"no comms")
#             parents.append(str(date))
#             ids.append(f"{date}-{slot}-no-comms")
#             colors.append(gray_color)
#             customdata.append(slot)
#             hovertemplate.append(f"<b>No comms</b><br><b>Time Slot:</b> {slot}")
#             line_colors.append("black")
#             line_widths.append(1)

# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     ids=ids,
#     branchvalues="total",
#     insidetextorientation='radial',
#     marker=dict(
#         colors=colors,
#         line=dict(color=line_colors, width=line_widths)
#     ),
#     customdata=customdata,
#     hovertemplate=hovertemplate,
#     sort=False
# ))

# fig.update_layout(
#     margin=dict(t=0, l=0, r=0, b=0),
#     title="Entities (Pairs) by Date with Clockwise Time Slots (No Comms in Gray, Highlighted Entities in Red)",
#     width=900,
#     height=900
# )

# fig.show()

In [27]:
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.colors
# from itertools import groupby
# from operator import itemgetter

# # Define all 15-min time slots (clockwise order)
# all_time_slots = pd.date_range("08:00", "15:00", freq="15min").strftime("%H:%M").tolist()[:-1][::-1]

# labels = []
# parents = []
# ids = []
# colors = []
# customdata = []
# hovertemplate = []

# gray_color = "lightgray"

# # Get unique dates and assign a unique color to each (clockwise)
# dates = sorted(output_df['date'].unique(), reverse=True)
# palette = plotly.colors.qualitative.Plotly
# while len(palette) < len(dates):
#     palette += palette

# date_color_map = {date: palette[i] for i, date in enumerate(dates)}

# # Root node
# labels.append("root")
# parents.append("")
# ids.append("root")
# colors.append("white")
# customdata.append("")
# hovertemplate.append("<b>All Dates</b>")

# for date in dates:
#     labels.append(str(date))
#     parents.append("root")
#     ids.append(str(date))
#     colors.append(date_color_map[date])
#     customdata.append("")
#     hovertemplate.append(f"<b>Date:</b> {date}")

#     df_date = output_df[output_df['date'] == date]
#     # Build a map: slot -> list of pairs
#     slot_to_pairs = {slot: [] for slot in all_time_slots}
#     for _, row in df_date.iterrows():
#         slots = [s.strip() for s in row['Time block'].strip('[]').split(',')]
#         for slot in slots:
#             if slot in slot_to_pairs:
#                 slot_to_pairs[slot].append(row['Pair'])

#     # Collect all pairs for this date (across all slots)
#     all_pairs = set()
#     for slot in all_time_slots:
#         all_pairs.update(slot_to_pairs[slot])

#     # Add each unique pair as a node (clockwise)
#     for pair in sorted(all_pairs, reverse=True):
#         # Find all slots this pair appears in (for hover)
#         pair_slots = [slot for slot in all_time_slots if pair in slot_to_pairs[slot]]
#         slot_str = ", ".join(sorted(pair_slots))
#         labels.append(pair)
#         parents.append(str(date))
#         ids.append(f"{date}-{pair}")
#         colors.append(date_color_map[date])
#         customdata.append(slot_str)
#         hovertemplate.append(f"<b>Pair:</b> {pair}<br><b>Time(s):</b> {slot_str}")

#     # Find all slots with no comms
#     no_comms_slots = [slot for slot in all_time_slots if not slot_to_pairs[slot]]
#     if no_comms_slots:
#         # Convert times to minutes for easier grouping
#         def time_to_min(t):
#             h, m = map(int, t.split(":"))
#             return h * 60 + m

#         slot_minutes = sorted([time_to_min(s) for s in no_comms_slots])
#         # Group consecutive slots
#         groups = []
#         for k, g in groupby(enumerate(slot_minutes), lambda x: x[0] - x[1]):
#             group = list(map(itemgetter(1), g))
#             groups.append(group)

#         # Build label string
#         slot_labels = []
#         for group in groups:
#             if len(group) == 1:
#                 # Single slot
#                 h, m = divmod(group[0], 60)
#                 slot_labels.append(f"{h:02d}:{m:02d}")
#             else:
#                 # Range
#                 h1, m1 = divmod(group[0], 60)
#                 h2, m2 = divmod(group[-1], 60)
#                 slot_labels.append(f"{h1:02d}:{m1:02d}–{h2:02d}:{m2:02d}")
#         time_range = ", ".join(slot_labels)

#         labels.append(f"no comms ({time_range})")
#         parents.append(str(date))
#         ids.append(f"{date}-no-comms-{time_range}")
#         colors.append(gray_color)
#         customdata.append(time_range)
#         hovertemplate.append(f"<b>No comms</b><br><b>Time(s):</b> {time_range}")

# fig = go.Figure(go.Sunburst(
#     labels=labels,
#     parents=parents,
#     ids=ids,
#     branchvalues="total",
#     insidetextorientation='radial',
#     marker=dict(colors=colors),
#     customdata=customdata,
#     hovertemplate=hovertemplate,
#     sort=False
# ))

# fig.update_layout(
#     margin=dict(t=0, l=0, r=0, b=0),
#     title="Entities (Pairs) by Date and 15-Minute Time Block (All No Comms Combined, Clockwise)",
#     width=900,
#     height=900
# )

# fig.show()